# 03 · Evaluate all models (v2): in-domain, real photos, synthetic faces and DFDC (cross-dataset)

Every trained model, the CLIP probe and every **CNN + CLIP** fused score is evaluated **independently and
identically** on the test split of every attached dataset:

| Test set | What it measures |
|---|---|
| **FF++**, **Celeb-DF** | in-domain detection (same kinds of data as training; held-out videos) |
| **FFHQ**, **CelebA-HQ** (real only) | false alarms on sharp, high-quality real photos (v1's main failure) |
| **StyleGAN**, **diffusion** (fake only) | fully AI-generated faces |
| **DFDC** | **cross-dataset generalisation**: never trained, early-stopped or calibrated on |

For two-class sets: AUC with **95% confidence intervals** (bootstrap over *videos*), accuracy and F1 at each
model's validation threshold, video-level AUC, AUC per manipulation, pairwise significance tests. For every set:
the **calibrated three-way verdict** (02c): coverage (share with a REAL/FAKE verdict), accuracy on those, FPR
(reals called FAKE), FNR (fakes called REAL), ECE before/after temperature scaling and the out-of-distribution
rate. Then **robustness curves** (blur / JPEG / resize) with the v1 **blur test**, and **v1 vs. v2** side by side.

Before anything is evaluated, the face crops are **audited for data leakage**.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** **01a**, **01b**, **01c**, **01d**, **02a** (+ copies), **02b**, **02c**; optionally the **v1**
  training output (old checkpoints) for the v1 vs. v2 comparison

**Produces:** `/kaggle/working/outputs/compare/` (+ `compare_v1/`), `robustness/` (+ `robustness_v1/`), `leakage_report.json`.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, find_datasets, checkpoint_roots, find_calibration, find_clip_probe
setup(data=False)

import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import display, Markdown, Image as Show
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve

ORDER = ["FF++", "CelebDF", "FFHQ", "CelebAHQ", "StyleGAN", "Diffusion", "DFDC"]   # DFDC last = cross-dataset
DATASETS = {k: v for k, v in sorted(find_datasets().items(), key=lambda kv: ORDER.index(kv[0]) if kv[0] in ORDER else 99)}

def version(root):
    p = next(Path(root).rglob("best.pt"), None)
    return torch.load(p, map_location="cpu", weights_only=False).get("version", 1) if p else None

ROOTS = [r for r in checkpoint_roots() if version(r) == 2]
V1_ROOTS = [r for r in checkpoint_roots() if version(r) == 1]
CAL, PROBE = find_calibration(), find_clip_probe()
MAIN = "efficientnet_b0"
print("datasets:", DATASETS)
print("v2 checkpoint folders:", ROOTS)
print("v1 checkpoint folders (reference):", V1_ROOTS)
print("calibration:", CAL, " CLIP probe:", PROBE)
assert "FF++" in DATASETS, "Add the output of 01a as an input"
assert ROOTS, "Add the output of 02a as an input"
if "DFDC" not in DATASETS:
    print("WARNING: DFDC (01d) not attached: no cross-dataset test.")
if CAL is None:
    print("WARNING: 02c not attached: no three-way verdict, no CLIP / fused rows.")

## Data-leakage audit
Runs on the actual face crops before any model is evaluated. **FAIL** = leakage (stop and fix before trusting
results); **WARN** = worth a look.
- every image on disk is in the manifest (no strays from earlier runs)
- no video / source image appears in more than one split; no FF++ identity in two splits
- Celeb-DF identities shared between train/val and the official test list: WARN (Celeb-DF's official protocol)
- no identical image in two splits or two datasets; near-duplicates via perceptual hash
- DFDC is test-only; class balance per split (two-class datasets)

In [ ]:
args = " ".join(f'--dataset "{k}={v}"' for k, v in DATASETS.items())
!TQDM_MININTERVAL=60 python -m src.classifier.check_leakage {args} --workers 4

## Run the evaluation
Each model is run on each test set; then the bootstrap CIs and pairwise tests are computed.

In [ ]:
roots = " ".join(f'"{r}"' for r in ROOTS)
datasets = " ".join(f'"{k}={v}"' for k, v in DATASETS.items())
extra = (f' --calibration "{CAL}"' if CAL else "") + (f' --clip-probe "{PROBE}"' if PROBE and CAL else "")
!TQDM_MININTERVAL=60 python -m src.classifier.compare --roots {roots} --datasets {datasets} --bootstrap 1000 --workers 4 {extra}

OUT = "/kaggle/working/outputs/compare"
DS_ALL = [d for d in DATASETS if os.path.isdir(f"{OUT}/{d}")]
comp = {d: pd.read_csv(f"{OUT}/{d}/comparison.csv").set_index("backbone") for d in DS_ALL}
DS = [d for d in DS_ALL if "auc" in comp[d]]          # two-class sets
preds = {d: pd.read_csv(f"{OUT}/{d}/predictions.csv") for d in DS_ALL}
per_method = {d: pd.read_csv(f"{OUT}/{d}/per_method.csv", index_col=0) for d in DS}
summary = pd.read_csv(f"{OUT}/summary.csv")

## Overview
`*_auc_ci` = 95% CI from the video bootstrap. `auc_drop` = FF++ AUC − DFDC AUC (lower = generalises better).
`*_fpr_3way` / `*_fnr_3way` / `*_coverage`: calibrated three-way verdict. Sorted by the last two-class set (DFDC).

In [ ]:
display(summary.round(3))
for fig in ("generalisation.png", "auc_vs_compute.png"):
    if os.path.exists(f"{OUT}/{fig}"): display(Show(f"{OUT}/{fig}", width=700))
for d in DS:
    display(Markdown(f"**ROC: all models on {d}**")); display(Show(f"{OUT}/{d}/roc_all.png", width=450))

## Three-way verdict (calibrated)
Per test set: `coverage` = share of images that got REAL or FAKE (the rest are UNCERTAIN), `acc_decided` = accuracy
on those, `fpr_3way` = real faces called FAKE, `fnr_3way` = fakes called REAL, `ece_raw → ece_calibrated` =
calibration error before/after temperature scaling, `ood_rate` = share flagged out of distribution.
The key v1 failures to check: **FAKE rate on FFHQ / CelebA-HQ** (should be ≈ 5% or lower) and **DFDC AUC**.

In [ ]:
COLS = ["coverage", "acc_decided", "fpr_3way", "fnr_3way", "ece_raw", "ece_calibrated", "ood_rate"]
for d in DS_ALL:
    t = comp[d]
    cols = [c for c in COLS if c in t and t[c].notna().any()]
    if cols:
        display(Markdown(f"### {d}")); display(t.set_index("model")[cols].round(3))

# Verdict matrix of the main model's fused score (or the CNN alone) on every set
key = f"fused:{MAIN}" if f"fused:{MAIN}__calibrated" in preds[DS_ALL[0]] else MAIN
if f"{key}__calibrated" in preds[DS_ALL[0]]:
    lo, hi = comp[DS_ALL[0]].loc[key, ["t_real", "t_fake"]]
    rows = {}
    for d in DS_ALL:
        p = preds[d]
        v = np.where(p[f"{key}__calibrated"] >= hi, "FAKE", np.where(p[f"{key}__calibrated"] < lo, "REAL", "UNCERTAIN"))
        for lab in sorted(p.label.unique()):
            s = pd.Series(v[p.label == lab]).value_counts(normalize=True)
            rows[(d, "fake" if lab else "real")] = s
    display(Markdown(f"**{comp[DS_ALL[0]].loc[key, 'model']}: verdict shares per test set and true label**"))
    display(pd.DataFrame(rows).T.reindex(columns=["REAL", "UNCERTAIN", "FAKE"]).fillna(0).round(3))

## Pairwise significance
Paired video bootstrap for every pair of models. `auc_diff_a_minus_b` > 0 means model A scored higher; the
difference is significant when the 95% CI excludes 0 (p < 0.05).

In [ ]:
for d in DS:
    sig = pd.read_csv(f"{OUT}/{d}/significance.csv")
    sig["significant"] = sig.p_value < 0.05
    display(Markdown(f"### {d}"))
    display(sig.sort_values("p_value").round(4))
    mat = sig.pivot(index="model_a", columns="model_b", values="p_value")
    display(Markdown("p-value matrix")); display(mat.round(3))

## Robustness and the blur test
Each model re-scored on 1,000 FF++ test images and on real photos (FFHQ, CelebA-HQ) at increasing blur, JPEG
compression and downscaling. **Blur test** (from the v1 diagnosis): the share of real faces called FAKE, sharp vs.
blurred at radius 1.5. v1 turned almost every blurred real photo FAKE.

In [ ]:
two = " ".join(f'"{k}={DATASETS[k]}"' for k in ("FF++", "CelebDF") if k in DATASETS)
reals = " ".join(f'"{k}={DATASETS[k]}"' for k in ("FFHQ", "CelebAHQ") if k in DATASETS)
cal = f'--calibration "{CAL}"' if CAL else ""
!TQDM_MININTERVAL=60 python -m src.classifier.robustness --roots {roots} {cal} --datasets {two} --reals {reals} --n 1000 --workers 4
R = "/kaggle/working/outputs/robustness"
display(pd.read_csv(f"{R}/blur_test.csv").round(3))
for kind in ("blur", "jpeg", "resample"):
    display(Show(f"{R}/robustness_{kind}.png", width=900))

## v1 vs. v2
Runs only when the v1 training output (the old FF++-only checkpoints) is attached: the same test sets and the
same blur test, at each v1 model's own threshold (v1 has no calibration).

In [ ]:
if V1_ROOTS:
    v1 = " ".join(f'"{r}"' for r in V1_ROOTS)
    !TQDM_MININTERVAL=60 python -m src.classifier.compare --roots {v1} --datasets {datasets} --bootstrap 200 --workers 4 --out /kaggle/working/outputs/compare_v1
    !TQDM_MININTERVAL=60 python -m src.classifier.robustness --roots {v1} --datasets {two} --reals {reals} --n 1000 --workers 4 --out /kaggle/working/outputs/robustness_v1
    s1 = pd.read_csv("/kaggle/working/outputs/compare_v1/summary.csv").set_index("backbone")
    s2 = summary.set_index("backbone")
    rows = {}
    for d in DS_ALL:
        c1 = pd.read_csv(f"/kaggle/working/outputs/compare_v1/{d}/comparison.csv").set_index("backbone")
        c2 = comp[d]
        metric = "auc" if "auc" in c2 else "accuracy"   # real-only / fake-only sets: share classified correctly
        for b in c1.index.intersection(c2.index):
            rows[(c2.loc[b, "model"], d)] = {"metric": metric, "v1": c1.loc[b, metric], "v2": c2.loc[b, metric]}
    display(pd.DataFrame(rows).T.round(3))
    display(Markdown("**Blur test, v1**")); display(pd.read_csv("/kaggle/working/outputs/robustness_v1/blur_test.csv").round(3))
else:
    print("v1 checkpoints not attached: skipped")

## Independent evaluation of each model
Same report for every model: metrics on the two-class test sets, AUC per manipulation, ROC and confusion matrices
at the model's own validation threshold, and its most confident mistakes on the cross-dataset set (DFDC).

In [ ]:
METRICS = ["auc", "auc_ci_low", "auc_ci_high", "accuracy", "f1", "video_auc", "threshold"]

def image_path(dataset, label, file):
    return f"{DATASETS[dataset]}/test/{'fake' if label == 1 else 'real'}/{file}"

def evaluate_model(backbone):
    name = comp[DS[0]].loc[backbone, "model"]
    per_method_ds = [d for d in DS if name in per_method[d]]
    cost = summary.set_index("backbone").loc[backbone]
    if pd.notna(cost.params_M):
        display(Markdown(f"### {name}\n{cost.params_M:.1f} M parameters · {cost.gflops:.2f} GFLOPs · "
                         f"{cost.images_per_s:.0f} images/s · trained in {cost.train_min or float('nan'):.0f} min"))
    else:
        display(Markdown(f"### {name}"))
    display(pd.DataFrame({d: comp[d].loc[backbone, METRICS] for d in DS}).T.astype(float).round(4))
    display(pd.concat({d: per_method[d][name] for d in per_method_ds}, axis=0).rename("AUC").to_frame().round(4))

    fig, axes = plt.subplots(1, 2 * len(DS), figsize=(5 * len(DS) * 2, 4))
    for i, d in enumerate(DS):
        p, thr = preds[d], comp[d].loc[backbone, "threshold"]
        fpr, tpr, _ = roc_curve(p.label, p[backbone])
        axes[2 * i].plot(fpr, tpr); axes[2 * i].plot([0, 1], [0, 1], "--", color="gray")
        axes[2 * i].set_title(f"{d} ROC (AUC {comp[d].loc[backbone, 'auc']:.3f})")
        axes[2 * i].set_xlabel("FPR"); axes[2 * i].set_ylabel("TPR")
        ConfusionMatrixDisplay(confusion_matrix(p.label, (p[backbone] > thr).astype(int)),
                               display_labels=["real", "fake"]).plot(ax=axes[2 * i + 1], cmap="Blues", colorbar=False)
        axes[2 * i + 1].set_title(f"{d} @ threshold {thr:.2f}")
    plt.tight_layout(); plt.show()

    d = DS[-1]  # most confident mistakes on the cross-dataset set (DFDC; the last two-class set attached)
    p, thr = preds[d], comp[d].loc[backbone, "threshold"]
    worst = pd.concat([p[(p.label == 0) & (p[backbone] > thr)].nlargest(5, backbone),
                       p[(p.label == 1) & (p[backbone] <= thr)].nsmallest(5, backbone)])
    if len(worst):
        fig, axes = plt.subplots(1, len(worst), figsize=(2.6 * len(worst), 3))
        for ax, r in zip(np.atleast_1d(axes), worst.itertuples()):
            ax.imshow(Image.open(image_path(d, r.label, r.file))); ax.axis("off")
            ax.set_title(f"{'real→fake' if r.label == 0 else 'fake→real'}\nP(fake)={getattr(r, backbone):.2f}", fontsize=8)
        plt.suptitle(f"{name}: most confident mistakes on {d}", fontsize=10); plt.tight_layout(); plt.show()

# Every CNN, the CLIP probe, and the main model's CNN + CLIP fusion
for backbone in summary.backbone:
    if not backbone.startswith("fused:") or backbone == f"fused:{MAIN}":
        evaluate_model(backbone)